# EDA geoespacial — subcuencas y red hidrográfica de Hermosillo

Exploración geoespacial del `subc_hermosillo.gpkg` (INEGI, red hidrográfica escala 1:50 000): las 5 capas de las cuatro subcuencas (La Manga, La Poza, R. San Miguel, R. Sonora-Hillo) unificadas en un solo archivo, con la columna `subcuenca` que identifica la subcuenca origen de cada fila.

Como polígono de la **ciudad de Hermosillo** se usa la unión de los asentamientos del DCAH 2025 con `locality_code == "0001"` (los 762 asentamientos de la ciudad; excluye Bahía de Kino `0137` y Miguel Alemán `0343`). Es una aproximación del área edificada, no el límite administrativo de la localidad.

In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
Subc_Hermosillo_gpkg = Path("../../data/processed/inegi/hidrografia_hermosillo/subc_hermosillo.gpkg")
Colonias_gpkg = Path("../../data/processed/aguah/colonias_hermosillo.gpkg")

capas = {
    nombre: gpd.read_file(Subc_Hermosillo_gpkg, layer=nombre)
    for nombre in gpd.list_layers(Subc_Hermosillo_gpkg)["name"]
}

## Resumen de las capas

El consolidado trae las mismas 5 capas que cada subcuenca por separado, unidas con la columna `subcuenca` de proveniencia.

In [ ]:
resumen = pd.DataFrame(
    {
        "filas": {k: len(v) for k, v in capas.items()},
        "geometria": {k: v.geom_type.unique().tolist() for k, v in capas.items()},
        "validas": {k: int(v.geometry.is_valid.sum()) for k, v in capas.items()},
        "crs": {k: str(v.crs) for k, v in capas.items()},
    }
)
resumen

## Polígono de la ciudad de Hermosillo

No hay un polígono oficial de la localidad en el proyecto, así que se aproxima el área edificada disolviendo los asentamientos del DCAH con `locality_code == "0001"`. El resultado se reproyecta a EPSG:4019 para operar con las capas hidrográficas.

In [ ]:
colonias = gpd.read_file(Colonias_gpkg, layer="colonias_hermosillo")
localidades_interes = ["0001"] # "0137", "0343"]

asentamientos_ciudad = colonias[colonias["locality_code"].isin(localidades_interes)]

hermosillo = gpd.GeoDataFrame(
    {"nombre": ["Hermosillo (union de asentamientos)"]},
    geometry=[asentamientos_ciudad.union_all()],
    crs=colonias.crs,
).to_crs(capas["subc"].crs)

print(f"{len(asentamientos_ciudad)} asentamientos disueltos en 1 poligono")

## Mapa 1: cobertura de la ciudad por subcuenca

Las cuatro unidades de captación (`subc`) frente al área de la ciudad de Hermosillo (puntos rojos)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

capas["subc"].plot(
    ax=ax, column="subcuenca", cmap="Purples",alpha=0.4, edgecolor="black", legend=True
)
hermosillo.plot(ax=ax, facecolor="none", edgecolor="red", linewidth=2)

ax.set_title("Unidades de captacion y el area edificada de Hermosillo")
ax.set_axis_off()
plt.show()

In [ ]:
print(hermosillo.geom_type)
print(hermosillo.geometry.iloc[0].geom_type)
print(hermosillo.geometry.iloc[0].is_valid)

crs = hermosillo.crs

print("¿Es proyectado?", crs.is_projected)
print("¿Es geográfico?", crs.is_geographic)
print("Unidades:", crs.axis_info[0].unit_name)

In [ ]:
hermosillo_m = hermosillo.to_crs(epsg=6372)

print(hermosillo_m.crs)
print(hermosillo_m.crs.is_projected)
print(hermosillo_m.crs.axis_info[0].unit_name)

## Mapa 2: la red hidrográfica dentro de la ciudad

`hl` y `ha` recortadas al polígono de la ciudad: la red que efectivamente cruza el área edificada.

In [ ]:
hl_ciudad = gpd.clip(capas["hl"], hermosillo)
ha_ciudad = gpd.clip(capas["ha"], hermosillo)

fig, ax = plt.subplots(figsize=(8, 8))

hermosillo.plot(ax=ax, facecolor="#f0f0f0", edgecolor="black", linewidth=1.5)
hl_ciudad.plot(ax=ax, column="subcuenca", linewidth=0.8, legend=True)
ha_ciudad.plot(ax=ax, color="deepskyblue", edgecolor="none")

ax.set_title("Red hidrografica dentro del area edificada de Hermosillo")
ax.set_axis_off()
plt.show()

## Mapa 3: red hidrográfica en contexto

El mismo encuadre del Mapa 2 (solo el área de Hermosillo) pero con las 5 capas: las unidades de captación (`subc`) de fondo, la red (`hl`) coloreada por subcuenca, los cuerpos de agua (`ha`) en azul, los puntos de drenaje (`dr`) y los topónimos (`to`).

In [ ]:
to_ciudad = gpd.clip(capas["to"], hermosillo)
to_ciudad.head()

In [ ]:
subc_ciudad = gpd.clip(capas["subc"], hermosillo)
dr_ciudad = gpd.clip(capas["dr"], hermosillo)
to_ciudad = gpd.clip(capas["to"], hermosillo)

fig, ax = plt.subplots(figsize=(8, 8))

subc_ciudad.plot(ax=ax, column="subcuenca", cmap="Purples", alpha=0.15, edgecolor="gray")
hl_ciudad.plot(ax=ax, color="deepskyblue", linewidth=0.8)
ha_ciudad.plot(ax=ax, color="deepskyblue", edgecolor="none")
dr_ciudad.plot(ax=ax, color="red", markersize=50)
to_ciudad.plot(ax=ax, column="CLASE", markersize=40, legend=True)
hermosillo.plot(ax=ax, facecolor="none", edgecolor="black", linewidth=0.5)

ax.set_title("Capas hidrograficas dentro del area edificada de Hermosillo")
ax.set_axis_off()
plt.show()

# Punto rojo es un punto de drenaje

## Cálculo de respaldo

Las cifras de los comentarios: kilómetros de red hidrográfica (`hl`) en la consolidada completa y dentro del polígono de la ciudad, medidos en EPSG:6372 (metros).

In [ ]:
km_total = capas["hl"].to_crs(epsg=6372).length.sum() / 1000
km_ciudad = hl_ciudad.to_crs(epsg=6372).length.sum() / 1000

metricas = pd.DataFrame(
    {
        "segmentos": [len(capas["hl"]), len(hl_ciudad)],
        "red_km": [km_total, km_ciudad],
    },
    index=["Consolidado (4 subcuencas)", "Dentro del area edificada"],
).round(1)
metricas

## Comentarios

- La red de las 4 subcuencas suma ~30,036 km de cauces (46,518 segmentos en las cuatro subcuencas). Dentro del área edificada hay ~189 km de red hidrográfica (437 segmentos, ~0.6% del total).
- La aproximación por asentamientos (localidad `0001`) es el área edificada, no el límite administrativo de la localidad: puede dejar huecos internos (parques, terrenos sin asentamiento) y excluir franjas edificadas sin polígono DCAH.

## GeoPackage de la ciudad: hidrogr_cd_hermosillo

Exporta las 5 capas de `subc_hermosillo` recortadas al área edificada de Hermosillo a `notebook_experimentation_data/hidrogr_cd_hermosillo.gpkg`, para reutilizarlas en exploraciones posteriores sin repetir el recorte.

In [ ]:
gpkg_ciudad = Path("notebook_experimentation_data/hidrogr_cd_hermosillo.gpkg")
gpkg_ciudad.parent.mkdir(parents=True, exist_ok=True)
gpkg_ciudad.unlink(missing_ok=True)

capas_ciudad = {
    "subc": subc_ciudad,
    "hl": hl_ciudad,
    "ha": ha_ciudad,
    "dr": dr_ciudad,
    "to": to_ciudad,
}

for nombre, gdf in capas_ciudad.items():
    gdf.to_file(gpkg_ciudad, layer=nombre, driver="GPKG")

pd.DataFrame({"filas": {k: len(v) for k, v in capas_ciudad.items()},},
    index=pd.Index(gpd.list_layers(gpkg_ciudad)["name"], name="capa"),
)